# 01: From text to next-token prediction

[English course index](../README_en.md) | [中文版本](../notebooks/01_tokenizer_and_lm.ipynb)

**Goal:** connect raw text, BPE tokens, shifted labels, cross-entropy, and perplexity.


## 1. What does a language model learn?

For tokens $x_{1:T}$, an autoregressive model factorizes the joint probability:

$$p_	heta(x_{1:T})=\prod_{t=1}^{T}p_	heta(x_t\mid x_{<t}).$$

The negative log-likelihood is

$$\mathcal L_{LM}=-rac{1}{T-1}\sum_{t=1}^{T-1}\log p_	heta(x_{t+1}\mid x_{\le t}).$$

Therefore input and target differ by one position: `[BOS,A,B] → [A,B,EOS]`. This off-by-one rule is shared by pretraining, SFT, and DPO.


### Hands-on check

Inspect shifted input and target using a tokenizer-independent toy sequence.


In [ ]:
toy_ids = [2, 10, 20, 3]  # BOS, A, B, EOS
toy_input = toy_ids[:-1]
toy_target = toy_ids[1:]
print("input: ", toy_input)
print("target:", toy_target)
assert toy_input[1:] == toy_target[:-1]


## 2. Why BPE?

Character sequences are long, while whole-word vocabularies cannot represent unseen words. BPE repeatedly merges the most frequent adjacent pair:

$$(a,b)^*=rg\max_{(a,b)}\operatorname{count}(a,b).$$

MiniLLM uses an 8,000-token vocabulary plus BOS/EOS/PAD tokens.

```bash
python tokenizer/train_tokenizer.py
```


### Hands-on check

Load the project tokenizer.


In [ ]:
from pathlib import Path
import sys, torch
ROOT = Path.cwd()
if not (ROOT / 'model').exists(): ROOT = ROOT.parents[2]
sys.path.insert(0, str(ROOT))
from tokenizer.tokenizer_utils import MiniLLMTokenizer
tok = MiniLLMTokenizer(str(ROOT / 'tokenizer/minillm_tokenizer.json'))


### Hands-on check

Encode and decode a short story sentence.


In [ ]:
text = 'Once upon a time, a little bird sang.'
ids = tok.encode(text, add_special_tokens=True)
print(ids)
print(tok.decode(ids, skip_special_tokens=True))
print('vocab =', tok.vocab_size)


### Hands-on check

Apply shifted labels to the real token sequence.


In [ ]:
input_ids = torch.tensor(ids[:-1])
targets = torch.tensor(ids[1:])
for x, y in zip(input_ids[:8], targets[:8]):
    print(repr(tok.decode([int(x)])), '->', repr(tok.decode([int(y)])))


## 3. Cross-entropy is negative log-likelihood

For logits $z\in\mathbb R^V$, softmax gives $p_i=e^{z_i}/\sum_j e^{z_j}$. With a one-hot target $q$:

$$H(q,p)=-\sum_iq_i\log p_i=-\log p_y.$$

PyTorch computes this without constructing one-hot vectors. `ignore_index=-100` masks prompt targets during SFT/DPO.


### Hands-on check

Compare `cross_entropy` with a manual `log_softmax + gather` implementation.


In [ ]:
import torch.nn.functional as F
T, V = len(targets), tok.vocab_size
logits = torch.randn(T, V)
loss = F.cross_entropy(logits, targets)
manual = -F.log_softmax(logits, dim=-1).gather(1, targets[:, None]).mean()
print('cross entropy:', loss.item())
print('manual NLL:   ', manual.item())
print('perplexity:   ', loss.exp().item())
assert torch.allclose(loss, manual)


## 4. Perplexity

For mean token NLL $L$:

$$\operatorname{PPL}=e^L.$$

Lower PPL means higher likelihood on the evaluation text. It does **not** by itself prove instruction following, factuality, or semantic quality.


## 5. Next step and source reading

- [Back to the English course index](../README_en.md)
- [Next: 02 Transformer](02_transformer.ipynb)
- [`tokenizer/tokenizer_utils.py`](../../../tokenizer/tokenizer_utils.py): runtime wrapper
- [`tokenizer/train_tokenizer.py`](../../../tokenizer/train_tokenizer.py): BPE training entry
- [`scripts/prepare_pretrain_data.py`](../../../scripts/prepare_pretrain_data.py): text-to-binary pipeline
- [`tests/test_tokenizer.py`](../../../tests/test_tokenizer.py): tokenizer contract tests
- [`tests/test_sft_labels.py`](../../../tests/test_sft_labels.py): shifted-label and mask regressions

Before moving on, explain in your own words why the target at position $t$ is the token at $t+1$.
